# Den 4: Zpracování textových dat – Cvičení: TF-IDF & Srovnání modelů (LR vs. SVM)

## Zadání úkolu:
1. Načtěte dataset `imdb_reviews_preprocessed.csv` pomocí knihovny **Pandas**.
2. Na základě sloupce `review_tokens_lemmatized` vytvořte numerickou reprezentaci textu pomocí vhodné třídy ze Scikit-learn (`TfidfVectorizer`).
3. Vytvořte DataFrame a rozdělte data na trénovací a testovací sadu (80 / 20).
4. Pomocí modelu **Logistické regrese** natrénujte binární klasifikátor sentimentu.
5. Natrénujte algoritmus na trénovacích datech.
6. Proveďte predikci na testovací sadě.
7. Vygenerujte `classification_report` a zhodnoťte výsledky modelu.
8. Zopakujte kroky 4–7 pro **Support Vector Machine (LinearSVC)** a porovnejte výsledky obou algoritmů.

### Krok 1: Import knihoven a načtení dat

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Načtení dat
data_path = Path("../data/imdb_reviews_preprocessed.csv")
if not data_path.exists():
    data_path = Path("data/imdb_reviews_preprocessed.csv")

imdb_reviews = pd.read_csv(data_path)
print(f"Načteno recenzí: {len(imdb_reviews)}")
imdb_reviews.head()

### Krok 2: Vektorizace pomocí `TfidfVectorizer`
Vytvoříme TF-IDF reprezentaci (Term Frequency - Inverse Document Frequency). Omezíme slovník na 10 000 nejvýznamnějších příznaků (`max_features=10000`).

In [ ]:
text_col = "review_tokens_lemmatized" if "review_tokens_lemmatized" in imdb_reviews.columns else "review_lemmatized"

vectorizer = TfidfVectorizer(max_features=10000)
X_tfidf = vectorizer.fit_transform(imdb_reviews[text_col].fillna(""))

print(f"Tvar TF-IDF matice: {X_tfidf.shape}")
print(f"Velikost slovníku: {len(vectorizer.get_feature_names_out()):,} termínů")

### Krok 3: Rozdělení dat na Train a Test (80 / 20)

In [ ]:
y = imdb_reviews["sentiment"].apply(lambda s: 1 if str(s).lower() == "positive" else 0)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Trénovací vzorky: {X_train.shape[0]}")
print(f"Testovací vzorky: {X_test.shape[0]}")

### Krok 4, 5, 6 & 7: Model 1 – Logistická regrese
Trénování, predikce a vyhodnocení classification reportu.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print(f"--- Logistická regrese (Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f} %) ---")
print(classification_report(y_test, y_pred_lr, target_names=["Negative (0)", "Positive (1)"]))

### Krok 8: Model 2 – Support Vector Machine (LinearSVC)
Trénování lineárního SVM a porovnání s Logistickou regresí.

In [ ]:
svm = LinearSVC(random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print(f"--- LinearSVC / SVM (Accuracy: {accuracy_score(y_test, y_pred_svm)*100:.2f} %) ---")
print(classification_report(y_test, y_pred_svm, target_names=["Negative (0)", "Positive (1)"]))

### Srovnání modelů a interpretace vah

In [ ]:
acc_lr = accuracy_score(y_test, y_pred_lr)
acc_svm = accuracy_score(y_test, y_pred_svm)

df_comp = pd.DataFrame({
    "Model": ["Logistic Regression", "LinearSVC (SVM)"],
    "Accuracy": [f"{acc_lr*100:.2f} %", f"{acc_svm*100:.2f} %"],
    "Správně klasifikováno": [f"{sum(y_test == y_pred_lr)} / {len(y_test)}", f"{sum(y_test == y_pred_svm)} / {len(y_test)}"]
})
df_comp